In [1]:
import os
import pandas as pd
import matplotlib as plt
import seaborn as sns


In [2]:
df_avito = pd.read_csv("Bronze/apparts_location.csv")

C:\Users\youssef\AppData\Local\Temp\ipykernel_24492\2742940154.py:1: DtypeWarning: Columns (0: Standing, 1: Condition) have mixed types. Specify dtype option on import or set low_memory=False.
  df_avito = pd.read_csv("Bronze/apparts_location.csv")


In [3]:
df_avito.info()

<class 'pandas.DataFrame'>
RangeIndex: 47101 entries, 0 to 47100
Data columns (total 36 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   description             47101 non-null  str    
 1   Frais de syndic / mois  19781 non-null  str    
 2   price                   38248 non-null  str    
 3   Meublé                  0 non-null      float64
 4   Chauffage               0 non-null      float64
 5   title                   47101 non-null  str    
 6   Concierge               0 non-null      float64
 7   category                0 non-null      float64
 8   price_value             38248 non-null  float64
 9   Standing                1 non-null      str    
 10  Surface totale          34036 non-null  str    
 11  Sécurité                0 non-null      float64
 12  area                    0 non-null      float64
 13  Surface habitable       45998 non-null  str    
 14  list_time               47101 non-null  str    
 

In [4]:
df_avito = df_avito.drop(
    columns=[
        
        'Meublé', 'Chauffage', 'Concierge', 'category', 'Sécurité',
        'Terrasse', 'Cuisine équipée', 'Balcon', 'Parking', 'WIFI','Caution',
        'Climatisation', 'Ascenseur','Frais de syndic / mois', 'Standing',
        'Surface totale', 'seller_name', 'seller_phone', 'image_count', 
        'Salons', 'Étage', "Type d'appartement", 'seller_type', 'image_url','price_value'
    ]
)

In [5]:
df_avito = df_avito.rename(columns={
    'Surface habitable': 'surface_m2',
    'list_time': 'list_date',
    'Chambres': 'rooms',
    'Condition': 'condition',
    'Salle de bain': 'bathrooms',
})

In [6]:
def to_number(col):
    return pd.to_numeric(col.astype(str).str.replace(r"\D", "", regex=True), errors="coerce")

df_avito["price"] = to_number(df_avito["price"])
df_avito["surface_m2"] = to_number(df_avito["surface_m2"])
df_avito["list_date"] = pd.to_datetime(df_avito["list_date"], errors="coerce", utc=True).dt.tz_localize(None)
df_avito["area"] = df_avito["area"].astype("object")

In [7]:
df_avito.info()

<class 'pandas.DataFrame'>
RangeIndex: 47101 entries, 0 to 47100
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   description  47101 non-null  str           
 1   price        38248 non-null  float64       
 2   title        47101 non-null  str           
 3   area         0 non-null      object        
 4   surface_m2   45351 non-null  float64       
 5   list_date    47101 non-null  datetime64[us]
 6   city         47101 non-null  str           
 7   rooms        46455 non-null  float64       
 8   condition    1 non-null      str           
 9   bathrooms    45460 non-null  float64       
 10  url          47101 non-null  str           
dtypes: datetime64[us](1), float64(4), object(1), str(5)
memory usage: 4.0+ MB


In [8]:
df_mubwab = pd.read_csv('Bronze/mubawab_rentals.csv')

In [9]:
import re
from urllib.parse import unquote

# === 1. Extract dates from image_names ===
def extract_date_from_images(img_names):
    if pd.isna(img_names):
        return None
    decoded = unquote(str(img_names))
    
    patterns = [
        r'WhatsApp.*?(\d{4})-(\d{2})-(\d{2})',
        r'IMG-(\d{4})(\d{2})(\d{2})',
        r'(\d{4})(\d{2})(\d{2})_\d{6}',
        r'Generated.*?(\d{4}).*?(\d{2}).*?(\d{2})',
        r'(\d{4})(\d{2})(\d{2})',
    ]
    
    for pat in patterns:
        m = re.search(pat, decoded, re.IGNORECASE)
        if m:
            try:
                groups = m.groups()
                if len(groups) == 3:
                    y, mth, d = map(int, groups)
                    if 2015 <= y <= 2027 and 1 <= mth <= 12 and 1 <= d <= 31:
                        return f"{y:04d}-{mth:02d}-{d:02d}"
            except:
                continue
    return None

df_mubwab['extracted_date'] = df_mubwab['image_names'].apply(extract_date_from_images)
df_mubwab['listing_date'] = pd.to_datetime(df_mubwab['extracted_date'], errors='coerce')

# === 2. Fill missing dates using ad_id ===
df_mubwab['ad_id_num'] = pd.to_numeric(df_mubwab['ad_id'], errors='coerce')
df_mubwab = df_mubwab.sort_values('ad_id_num').reset_index(drop=True)

df_mubwab['listing_date_filled'] = df_mubwab['listing_date'].copy()
df_mubwab['listing_date_filled'] = df_mubwab['listing_date_filled'].interpolate(
    method='linear', 
    limit_direction='both'
)

# Final date
df_mubwab['listing_date'] = df_mubwab['listing_date'].combine_first(df_mubwab['listing_date_filled'])

# === NEW: Boolean column ===
df_mubwab['is_extracted'] = df_mubwab['listing_date'].notna()



In [10]:
split = df_mubwab['location'].str.split(r'\s*,+\s*', n=1, expand=True, regex=True)

df_mubwab['area'] = split[0].str.strip()
df_mubwab['city'] = split[1].str.strip()

In [11]:
df_mubwab = df_mubwab.drop(
    columns=[
        'location', 'ad_id', 'listing_type', 'orientation', 'floor_type',
        'garden_m2', 'garage_spaces', 'features_text', 'phone',
        'agency_name', 'agency_url', 'image_names', 'page', 'extracted_date',
        'listing_date', 'ad_id_num', 'is_extracted', 'bedrooms'
        
    ]
)

In [12]:
df_mubwab = df_mubwab.rename(columns={
    'price_DH': 'price',
    'listing_date_filled': 'list_date',
})

In [13]:

df_mubwab["price"] = to_number(df_mubwab["price"])

In [14]:
df_mubwab.info()

<class 'pandas.DataFrame'>
RangeIndex: 13266 entries, 0 to 13265
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   title        13266 non-null  str           
 1   price        13011 non-null  float64       
 2   surface_m2   12790 non-null  float64       
 3   rooms        12661 non-null  float64       
 4   bathrooms    5501 non-null   float64       
 5   condition    6278 non-null   str           
 6   description  6999 non-null   str           
 7   url          13266 non-null  str           
 8   list_date    13266 non-null  datetime64[us]
 9   area         13266 non-null  str           
 10  city         13213 non-null  str           
dtypes: datetime64[us](1), float64(4), str(6)
memory usage: 1.1 MB


In [15]:
df_mubwab.sample(10)

,title,price,surface_m2,rooms,bathrooms,condition,description,url,list_date,area,city
5511,Appartement meublé à louer,10000.0,96.0,2.0,NaN,NaN,NaN,https://www.mubawab.ma/fr/a/8253214/appartemen...,2025-11-14 00:00:00,Agdal,Rabat
8917,Superbe appartement à louer en Centre ville. S...,200.0,90.0,1.0,1.0,NaN,NaN,https://www.mubawab.ma/fr/a/8306406/superbe-ap...,2026-02-07 08:00:00,Centre ville,Nador
11060,"Appartement à louer – 2 chambres, 2 salles de ...",16000.0,100.0,3.0,NaN,Projet neuf,Découvrez ce magnifique appartement idéalement...,https://www.mubawab.ma/fr/a/8327476/appartemen...,2025-12-15 00:00:00,De La Plage,Tanger
3371,À louer bel appartement climatisé avec piscine,750.0,80.0,3.0,1.0,NaN,NaN,https://www.mubawab.ma/fr/a/8207795/%C3%A0-lou...,2025-09-22 16:00:00,Saïdia,Saïdia
9418,Appartement 210 m2 3 chambres 3 salons Hay Riad,17000.0,220.0,3.0,NaN,Jamais habité / rénové,Caractéristiques principales et agencement Ap...,https://www.mubawab.ma/fr/a/8311532/appartemen...,2025-07-14 00:00:00,Riyad,Rabat
2274,"Appartement à Louer de 133.0 m² à Casablanca, ...",15500.0,133.0,5.0,NaN,NaN,"À louer, charmant appartement de 123 m² situé ...",https://www.mubawab.ma/fr/a/8160508/appartemen...,2025-11-06 12:00:00,Ain Diab,Casablanca
12741,Maison propre et lumineuse à louer à Hay Anas,3000.0,83.0,1.0,1.0,Projet neuf,Je mets en location une belle maison adaptée a...,https://www.mubawab.ma/fr/a/8339293/maison-pro...,2023-06-25 00:00:00,Route ain Chkaf,Fès
4662,Appartement sans vis a vis à louer à oasis 3 c...,12000.0,112.0,4.0,NaN,NaN,NaN,https://www.mubawab.ma/fr/a/8237109/appartemen...,2025-12-24 19:12:00,Riviera,Casablanca
12382,Studio Meublé Haut Standing Avec Piscine Major...,8500.0,45.0,1.0,1.0,Jamais habité / rénové,"Bonjour, nous présentons un studio haut standi...",https://www.mubawab.ma/fr/a/8336977/studio-meu...,2026-04-21 00:00:00,Majorelle,Marrakech
5757,Louer Appartement meublé façade à Mimosas Kénitra,4800.0,90.0,4.0,1.0,NaN,NaN,https://www.mubawab.ma/fr/a/8258446/louer-appa...,2025-11-25 00:00:00,Mimosas,Kénitra


In [16]:
df_avito["listing_type"] = "rent"
df_mubwab["listing_type"] = "rent"

df_merged = pd.concat([df_avito, df_mubwab])

In [17]:
df_merged.to_csv("Silver/rent_clean.csv")

In [18]:
df_merged

,description,price,title,area,surface_m2,list_date,city,rooms,condition,bathrooms,url,listing_type
0,"Appartement à louer – Quartier Victor Hugo, 1e...",10500.0,Appartements à louer 84 m² à Marrakech,NaN,90.0,2026-06-03 08:59:52,Marrakech,2.0,NaN,2.0,https://www.avito.ma/fr/gu%C3%A9liz/appartemen...,rent
1,Boulevard d'Anfa. Appartements et Studios Neuf...,15000.0,APPARTEMENTS et STUDIOS HAUT STANDING Bd D'Anfa,NaN,70.0,2026-05-31 09:36:42,Casablanca,2.0,NaN,2.0,https://www.avito.ma/fr/anfa/appartements/APPA...,rent
2,Serraj Immobilier vous propose: un joli studio...,7500.0,Appartement à louer 50 m² à Casablanca,NaN,50.0,2026-05-25 08:59:48,Casablanca,1.0,NaN,1.0,https://www.avito.ma/fr/bourgogne/appartements...,rent
3,"À louer – Appartement à Hivernage, Marrakech 📍...",10500.0,Appartement moderne 2 chambres à louer l’hiver...,NaN,65.0,2026-06-03 09:08:42,Marrakech,2.0,NaN,2.0,https://www.avito.ma/fr/hivernage/appartements...,rent
4,Serraj Immobilier vous propose : Bel apparteme...,8500.0,Appartement à louer 114 m² à Almaz,NaN,114.0,2026-06-02 19:02:13,Casablanca,2.0,NaN,2.0,https://www.avito.ma/fr/almaz/appartements/App...,rent
...,...,...,...,...,...,...,...,...,...,...,...,...
13261,Nouvelle annonce de Location d'un appartement ...,8000.0,"Appartement à Louer de 78.0 m² à Marrakech, Ba...",Bab Ighli,78.0,2026-04-03 08:00:00,Marrakech,4.0,NaN,NaN,https://www.mubawab.ma/fr/a/8342379/appartemen...,rent
13262,"À louer, charmant appartement de 110 m² situé ...",11500.0,"Appartement à Louer de 110.0 m² à Casablanca, ...",Racine,110.0,2026-03-30 00:00:00,Casablanca,4.0,NaN,NaN,https://www.mubawab.ma/fr/a/8342381/appartemen...,rent
13263,Nouvelle annonce de Location d'un appartement ...,14000.0,"Appartement à Louer de 147.0 m² à Casablanca, ...",Palmier,147.0,2025-12-12 00:00:00,Casablanca,5.0,NaN,NaN,https://www.mubawab.ma/fr/a/8342384/appartemen...,rent
13264,"À louer, bel appartement de 60 m² situé à Bab ...",7000.0,"Appartement à Louer de 60.0 m² à Marrakech, Hi...",Hivernage,60.0,2026-04-29 00:00:00,Marrakech,4.0,NaN,NaN,https://www.mubawab.ma/fr/a/8342385/appartemen...,rent
